# Inverse-RL Colab Notebook

Single resumable notebook for the inverse-RL experiment. Cells 0–3 are implemented; cells 4–8 are intentionally left as labeled stubs for later execution-guide steps.

In [1]:
# Cell 0 — setup (repo, branch, dependencies, GPU)

import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
os.environ["VLLM_USE_V1"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import subprocess
from pathlib import Path
from importlib import metadata

# =========================
# CONFIG
# =========================

GIT_URL = os.environ.get(
    "INVERSE_RL_GIT_URL",
    "https://github.com/shengweiming/inverse-RL.git",
)

BRANCH = "codex/create-inverse_rl_colab.ipynb-with-resumable-cells"  # <-- change branch here when needed

REPO_DIR = Path("/content/inverse-RL")

# vLLM version known to work on Colab CUDA 12.8
INSTALL_VLLM = True
VLLM_VERSION = "0.10.2"
FORCE_REINSTALL_VLLM = False  # set True if vLLM breaks again

def run(cmd):
    print("[run]", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

# =========================
# CLONE REPO IF NEEDED
# =========================

if REPO_DIR.exists():
    print(f"[skip] repo already exists at {REPO_DIR}")
else:
    print(f"[run] cloning {GIT_URL} -> {REPO_DIR}")
    run(["git", "clone", GIT_URL, str(REPO_DIR)])

# =========================
# MOVE INTO REPO
# =========================

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# =========================
# FETCH + CHECKOUT BRANCH
# =========================

print("[run] fetching latest branches")
run(["git", "fetch", "--all"])

print(f"[run] checking out branch: {BRANCH}")
run(["git", "checkout", BRANCH])

print("[run] pulling latest changes")
run(["git", "pull"])

# =========================
# SHOW REPO INFO
# =========================

print("[run] current branch:")
run(["git", "branch", "--show-current"])

print("[run] current commit:")
run(["git", "rev-parse", "HEAD"])

print("[run] repo files:")
run(["ls"])

# =========================
# INSTALL REQUIREMENTS
# =========================

requirements_path = REPO_DIR / "requirements.txt"

if requirements_path.exists():
    print("[run] installing requirements")
    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ])
else:
    print("[skip] no requirements.txt found")

# =========================
# INSTALL vLLM COMPATIBLY FOR COLAB
# =========================

if INSTALL_VLLM:
    try:
        installed_vllm = metadata.version("vllm")
    except metadata.PackageNotFoundError:
        installed_vllm = None

    needs_vllm_install = (
        FORCE_REINSTALL_VLLM
        or installed_vllm != VLLM_VERSION
    )

    if needs_vllm_install:
        print(f"[run] installing vLLM=={VLLM_VERSION} with CUDA 12.8 backend")

        run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "vllm-flash-attn"])
        run([sys.executable, "-m", "pip", "cache", "purge"])
        run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
        run(["uv", "pip", "install", "--system", f"vllm=={VLLM_VERSION}", "--torch-backend=cu128"])

        print("[note] If this is the first vLLM install in this runtime, restart runtime before importing vLLM.")
    else:
        print(f"[skip] vLLM=={VLLM_VERSION} already installed")

# =========================
# PIN TOKENIZER STACK FOR vLLM 0.10.2
# =========================
print("[run] pinning transformers/tokenizers for vLLM compatibility")
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "transformers==4.56.2",
    "tokenizers==0.22.1",
    "huggingface-hub>=0.34.0",
])

# =========================
# GPU INFO
# =========================

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

[skip] repo already exists at /content/inverse-RL
[run] fetching latest branches
[run] git fetch --all
[run] checking out branch: codex/create-inverse_rl_colab.ipynb-with-resumable-cells
[run] git checkout codex/create-inverse_rl_colab.ipynb-with-resumable-cells
[run] pulling latest changes
[run] git pull
[run] current branch:
[run] git branch --show-current
[run] current commit:
[run] git rev-parse HEAD
[run] repo files:
[run] ls
[run] installing requirements
[run] /usr/bin/python3 -m pip install -q -r /content/inverse-RL/requirements.txt
[skip] vLLM==0.10.2 already installed
[run] pinning transformers/tokenizers for vLLM compatibility
[run] /usr/bin/python3 -m pip install -q --force-reinstall transformers==4.56.2 tokenizers==0.22.1 huggingface-hub>=0.34.0
NVIDIA L4, 23034 MiB


In [2]:
# Cell 1 — Drive + config + resumable state
import json
import os
from pathlib import Path

from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/inverse-rl")
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "ckpts"
RESULTS_DIR = ROOT / "results"
for d in (ROOT, DATA_DIR, CKPT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    "model_name": "meta-llama/Llama-3.2-3B-Instruct",
    "shakedown_model": "meta-llama/Llama-3.2-1B-Instruct",
    "lora_r": 32,
    "G": 8,
    "max_prompt_len": 2048,
    "max_completion_len": 512,
    "lr": 2e-6,
    "train_levels": [1],
    "composition_train_levels": [2],
    "eval_levels": [1, 2, 3, 4],
    "pass_at_k": 8,
    "rollout_k": 4,
    "rft_k": 4,
    "sft_epochs": 1,
    "g2_pass1_threshold": 0.30,
    "seeds": [0, 1, 2],
    # Dataset sizes are intentionally configurable for shakedowns vs. full runs.
    "n_forward_l1": 2500,
    "n_eval_per_cell": 100,
    "n_comp_train": 2500,
    "n_inverse_train": 2500,
    "n_inverse_l1to2_seen": 1200,
}

HF_TOKEN = userdata.get("HF_TOKEN")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("[skip] HF_TOKEN not found in Colab userdata; gated model downloads may fail")

STATE_PATH = ROOT / "state.json"

def state_load():
    if STATE_PATH.exists():
        with STATE_PATH.open("r", encoding="utf-8") as f:
            return json.load(f)
    return {}

state = state_load()

def state_save(key=None, value=True):
    global state
    state = state_load()
    if key is not None:
        state[key] = value
    tmp = STATE_PATH.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, sort_keys=True)
    tmp.replace(STATE_PATH)
    return state

def exists(path):
    return Path(path).exists()

def done(key):
    return bool(state_load().get(key, False))

state_save()
print(f"ROOT={ROOT}")
print(f"state={STATE_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT=/content/drive/MyDrive/inverse-rl
state=/content/drive/MyDrive/inverse-rl/state.json


In [3]:
# Cell 2 — data generation (resumable JSONL outputs)
import subprocess
import sys
from pathlib import Path

DATASETS = {
    "fwd_l1_all.jsonl": ["--task", "forward", "--levels", "1", "--n", str(CFG["n_forward_l1"]), "--pool", "all", "--seed", str(CFG["seeds"][0])],
    "fwd_l1to4_eval.jsonl": ["--task", "forward", "--levels", "1,2,3,4", "--n", str(CFG["n_eval_per_cell"]), "--pool", "all", "--eval", "--seed", str(CFG["seeds"][0])],
    "fwd_l2_seen_comp_train.jsonl": ["--task", "forward", "--levels", "2", "--n", str(CFG["n_comp_train"]), "--pool", "seen", "--seed", str(CFG["seeds"][1])],
    "inv_l1_seen_train.jsonl": ["--task", "inverse", "--levels", "1", "--n", str(CFG["n_inverse_train"]), "--pool", "seen", "--seed", str(CFG["seeds"][1])],
    "inv_l1to4_eval.jsonl": ["--task", "inverse", "--levels", "1,2,3,4", "--n", str(CFG["n_eval_per_cell"]), "--pool", "all", "--eval", "--seed", str(CFG["seeds"][2])],
    "inv_l1to2_seen_optional.jsonl": ["--task", "inverse", "--levels", "1,2", "--n", str(CFG["n_inverse_l1to2_seen"]), "--pool", "seen", "--seed", str(CFG["seeds"][2])],
}

missing = [name for name in DATASETS if not (DATA_DIR / name).exists()]
if done("data") and not missing:
    print(f"[skip] data already present in {DATA_DIR}")
else:
    print(f"[run] generating {len(missing)} missing dataset(s) in {DATA_DIR}")
    for name in missing:
        out = DATA_DIR / name
        cmd = [sys.executable, "inverse_tasks.py", *DATASETS[name], "--out", str(out)]
        print("[run]", " ".join(cmd))
        subprocess.run(cmd, check=True)
    state_save("data", True)
    print("[run] data generation complete")


[skip] data already present in /content/drive/MyDrive/inverse-rl/data


In [4]:
# Cell 3 — evaluation utilities (HF/LoRA loading, vLLM generation, CSV outputs)
import gc
import json
import re
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

from inverse_tasks import ALL_SKILLS
from skills_inverse import SKILLS
from verifier import forward_reward, inverse_reward

_ACTIVE_MODEL = None
_ACTIVE_LLM = None
_ACTIVE_LLM_KEY = None

def _safe_name(path_or_name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(path_or_name).rstrip("/"))[-120:]

def _read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def _model_key(model):
    if isinstance(model, dict):
        return model["model_path"]
    return str(model)

def load_model(path_or_name, adapter=None):
    """Return a vLLM-ready model descriptor; merge an optional LoRA adapter once."""
    if adapter is None:
        model = {"model_path": str(path_or_name), "adapter": None, "name": _safe_name(path_or_name)}
        print(f"[skip] using base/merged model {model['model_path']}")
        return model

    merged_dir = CKPT_DIR / f"merged_{_safe_name(path_or_name)}__{_safe_name(adapter)}"
    if merged_dir.exists():
        print(f"[skip] merged adapter already exists at {merged_dir}")
        return {"model_path": str(merged_dir), "adapter": str(adapter), "name": _safe_name(merged_dir)}

    print(f"[run] merging LoRA adapter {adapter} into {path_or_name}")
    tokenizer = AutoTokenizer.from_pretrained(path_or_name, token=HF_TOKEN)
    base = AutoModelForCausalLM.from_pretrained(
        path_or_name,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    merged = PeftModel.from_pretrained(base, adapter).merge_and_unload()
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    del merged, base, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"model_path": str(merged_dir), "adapter": str(adapter), "name": _safe_name(merged_dir)}

def _get_llm(model):
    global _ACTIVE_LLM, _ACTIVE_LLM_KEY
    key = _model_key(model)
    if _ACTIVE_LLM is not None and _ACTIVE_LLM_KEY == key:
        return _ACTIVE_LLM
    if _ACTIVE_LLM is not None:
        del _ACTIVE_LLM
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"[run] loading vLLM model {key}")
    _ACTIVE_LLM = LLM(
        model=key,
        tokenizer=key,
        trust_remote_code=True,
        dtype="bfloat16",
        max_model_len=CFG["max_prompt_len"] + CFG["max_completion_len"],
        gpu_memory_utilization=0.60,
        enforce_eager=True,
      )
    _ACTIVE_LLM_KEY = key
    return _ACTIVE_LLM

def generate(prompts, n=1, model=None):
    """Generate n completions per prompt with vLLM; call eval_task/load_model first or pass model."""
    global _ACTIVE_MODEL
    if model is not None:
        _ACTIVE_MODEL = model
    if _ACTIVE_MODEL is None:
        raise ValueError("No active model. Pass model=... or call eval_task(model, ...) first.")
    llm = _get_llm(_ACTIVE_MODEL)
    params = SamplingParams(
        n=n,
        temperature=0.7 if n > 1 else 0.0,
        top_p=0.95,
        max_tokens=CFG["max_completion_len"],
    )
    outputs = llm.generate(prompts, params)
    return [[choice.text for choice in out.outputs] for out in outputs]

def _cells_from_arg(cells):
    if isinstance(cells, (str, Path)):
        return _read_jsonl(cells)
    return list(cells)

def eval_task(model, cells, task, k=None, out_name=None):
    """Evaluate a task and save tidy metrics: level, split, pass@1, pass@k."""
    global _ACTIVE_MODEL
    k = int(k or CFG["pass_at_k"])
    problems = _cells_from_arg(cells)
    model_name = model.get("name", _safe_name(_model_key(model))) if isinstance(model, dict) else _safe_name(model)
    out_csv = RESULTS_DIR / (out_name or f"eval_{task}_{model_name}_k{k}.csv")
    if out_csv.exists():
        print(f"[skip] loading existing eval results from {out_csv}")
        return pd.read_csv(out_csv)

    print(f"[run] evaluating {task} on {len(problems)} problems with k={k}")
    _ACTIVE_MODEL = model
    reward_fn = forward_reward if task == "forward" else inverse_reward
    grouped = {}
    prompts = [p["prompt"] for p in problems]
    completions = generate(prompts, n=k)
    for problem, samples in zip(problems, completions):
        rewards = [reward_fn(sample, problem) for sample in samples]
        level = int(problem.get("level", 0))
        split = problem.get("eval_split") or ("seen" if problem.get("skills_seen") else "held_out")
        key = (level, split)
        grouped.setdefault(key, {"p1": [], "pk": []})
        grouped[key]["p1"].append(float(rewards[0] == 1.0) if rewards else 0.0)
        grouped[key]["pk"].append(float(any(r == 1.0 for r in rewards)))

    rows = []
    for (level, split), vals in sorted(grouped.items()):
        rows.append({
            "level": level,
            "split": split,
            "pass@1": sum(vals["p1"]) / len(vals["p1"]),
            f"pass@{k}": sum(vals["pk"]) / len(vals["pk"]),
            "n": len(vals["p1"]),
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    state_save(out_csv.stem, True)
    print(f"[run] saved {out_csv}")
    return df

def forward_accuracy(model, eval_jsonl=None, k=1, out_name=None):
    """Forward pass@1 by skill and tier on level-1 forward eval examples."""
    eval_jsonl = eval_jsonl or (DATA_DIR / "fwd_l1to4_eval.jsonl")
    problems = [p for p in _read_jsonl(eval_jsonl) if int(p.get("level", 0)) == 1]
    model_name = model.get("name", _safe_name(_model_key(model))) if isinstance(model, dict) else _safe_name(model)
    out_csv = RESULTS_DIR / (out_name or f"forward_accuracy_{model_name}.csv")
    if out_csv.exists():
        print(f"[skip] loading existing forward accuracy from {out_csv}")
        return pd.read_csv(out_csv)

    print(f"[run] computing forward accuracy for {len(problems)} level-1 problems")
    global _ACTIVE_MODEL
    _ACTIVE_MODEL = model
    completions = generate([p["prompt"] for p in problems], n=k)
    rows = []
    for problem, samples in zip(problems, completions):
        skill = problem["chain"][0]
        tier = SKILLS[skill][4]
        rewards = [forward_reward(sample, problem) for sample in samples]
        rows.append({
            "skill": skill,
            "tier": tier,
            "correct": float(any(r == 1.0 for r in rewards)),
        })
    df = pd.DataFrame(rows).groupby(["skill", "tier"], as_index=False).agg(accuracy=("correct", "mean"), n=("correct", "size"))
    df.to_csv(out_csv, index=False)
    state_save(out_csv.stem, True)
    print(f"[run] saved {out_csv}")
    return df

print("[skip] eval utilities are defined; call load_model(), eval_task(), and forward_accuracy() as needed")


INFO 06-02 21:39:15 [__init__.py:216] Automatically detected platform cuda.
[skip] eval utilities are defined; call load_model(), eval_task(), and forward_accuracy() as needed


In [8]:
stage_0_smoke_test = False

if stage_0_smoke_test == True:

  model = load_model(CFG["shakedown_model"])

  tiny = _read_jsonl(DATA_DIR / "inv_l1to4_eval.jsonl")[:8]

  df = eval_task(
      model=model,
      cells=tiny,
      task="inverse",
      k=1,
      out_name="smoke_inv_1b_k1_tiny.csv",
  )

  df

In [ ]:
# Cell 4 — Stage 1 forward SFT on paper Stage-1 corpus
import gc
import json
import re
from pathlib import Path

import pandas as pd
import torch
import wandb
from datasets import Dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig
from prompts import extract_last_json
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer

STAGE1_DIR = CKPT_DIR / "stage1_sft"
STAGE1_ADAPTER_DIR = CKPT_DIR / "stage1_sft_adapter"
STAGE1_STATS_CSV = STAGE1_DIR / "stage1_sft_stats.csv"
STAGE1_FORWARD_CSV = STAGE1_DIR / "stage1_forward.csv"
FILTERED_TRAIN_PARQUET = STAGE1_DIR / "filtered_train.parquet"
FILTERED_TEST_PARQUET = STAGE1_DIR / "filtered_test.parquet"
PAPER_STAGE1_REPO = "weizechen/RL-Compositionality-Stage1-RFT-Data"
FUNC_RE = re.compile(r"\bfunc_(\d+)\s*\(")
PAPER_FUNC_TO_SKILL = {
    "1": "repeat_str",
    "4": "reverse_words",
    "5": "add_prefix",
    "6": "add_suffix",
    "8": "rotate_str",
    "9": "mirror_str",
    "13": "insert_separator",
    "14": "duplicate_every_char",
    "15": "fancy_brackets",
}


def _stage1_text(value):
    """Return plain text from string, chat-message, or parquet object values."""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, dict):
        if "content" in value:
            return str(value["content"]).strip()
        return json.dumps(value, ensure_ascii=False).strip()
    if hasattr(value, "tolist"):
        return _stage1_text(value.tolist())
    if isinstance(value, (list, tuple)):
        pieces = []
        for item in value:
            if isinstance(item, dict) and "content" in item:
                pieces.append(str(item["content"]))
            else:
                pieces.append(_stage1_text(item))
        return "\n".join(piece for piece in pieces if piece).strip()
    return str(value).strip()


def _stage1_prompt(row):
    if "prompt" not in row.index:
        raise KeyError(f"paper Stage-1 parquet is missing a prompt column; columns={list(row.index)}")
    return _stage1_text(row["prompt"])


def _stage1_has_value(value):
    marker = pd.notna(value)
    return bool(marker.all()) if hasattr(marker, "all") else bool(marker)


def _stage1_response(row):
    for column in ("response", "completion", "answer", "output"):
        if column in row.index and _stage1_has_value(row[column]):
            return _stage1_text(row[column])
    raise KeyError(f"paper Stage-1 parquet is missing a response/completion column; columns={list(row.index)}")


def _stage1_func_id(prompt):
    matches = FUNC_RE.findall(prompt)
    return matches[0] if len(matches) == 1 else None


def _reward_ground_truth(row):
    if "reward_model" not in row.index:
        raise KeyError(f"paper Stage-1 parquet is missing reward_model; columns={list(row.index)}")
    reward_model = row.get("reward_model") if hasattr(row, "get") else row["reward_model"]
    if hasattr(reward_model, "as_py"):
        reward_model = reward_model.as_py()
    if isinstance(reward_model, str):
        try:
            reward_model = json.loads(reward_model)
        except json.JSONDecodeError:
            return reward_model
    if isinstance(reward_model, dict):
        ground_truth = reward_model.get("ground_truth")
    else:
        ground_truth = getattr(reward_model, "ground_truth", None)
    if isinstance(ground_truth, str):
        try:
            parsed = json.loads(ground_truth)
        except json.JSONDecodeError:
            return ground_truth
        if isinstance(parsed, dict) and "output" in parsed:
            return parsed["output"]
        return ground_truth
    if isinstance(ground_truth, dict) and "output" in ground_truth:
        return ground_truth["output"]
    return ground_truth


def _require_hidden_def_prompts(df, split_name):
    bad = df[df["stripped_prompt"].str.contains(r"def\s+func_\d+\s*\(", regex=True, na=False)]
    if len(bad):
        raise ValueError(f"{split_name} has {len(bad)} prompts with visible func_N definitions; refusing contaminated Stage-1 SFT")


def _load_and_filter_stage1_sft_data():
    if FILTERED_TRAIN_PARQUET.exists() and FILTERED_TEST_PARQUET.exists() and STAGE1_STATS_CSV.exists():
        print(f"[skip] loading filtered paper Stage-1 SFT data from {STAGE1_DIR}")
        train_df = pd.read_parquet(FILTERED_TRAIN_PARQUET)
        test_df = pd.read_parquet(FILTERED_TEST_PARQUET)
        stats_df = pd.read_csv(STAGE1_STATS_CSV)
        print(stats_df.to_string(index=False))
        print(f"Filtered Stage-1 train total: {len(train_df)}")
        return train_df, test_df, stats_df

    print(f"[run] downloading train/test parquet from {PAPER_STAGE1_REPO}")
    train_path = hf_hub_download(PAPER_STAGE1_REPO, filename="train.parquet", repo_type="dataset", token=HF_TOKEN)
    test_path = hf_hub_download(PAPER_STAGE1_REPO, filename="test.parquet", repo_type="dataset", token=HF_TOKEN)
    raw_train = pd.read_parquet(train_path)
    raw_test = pd.read_parquet(test_path)

    def prepare(df, split_name, cap_per_skill=None, require_response=False):
        prepared = df.copy()
        prepared["stripped_prompt"] = prepared.apply(_stage1_prompt, axis=1)
        prepared["func_id"] = prepared["stripped_prompt"].map(_stage1_func_id)
        prepared = prepared[prepared["func_id"].isin(PAPER_FUNC_TO_SKILL)].copy()
        prepared["skill"] = prepared["func_id"].map(PAPER_FUNC_TO_SKILL)
        response_columns = {"response", "completion", "answer", "output"} & set(prepared.columns)
        if require_response or response_columns:
            prepared["response"] = prepared.apply(_stage1_response, axis=1)
        _require_hidden_def_prompts(prepared, split_name)
        if cap_per_skill is not None:
            capped = []
            for _func_id, group in prepared.groupby("func_id", sort=True):
                if len(group) > cap_per_skill:
                    group = group.sample(n=cap_per_skill, random_state=42)
                capped.append(group)
            prepared = pd.concat(capped, ignore_index=True) if capped else prepared.iloc[0:0].copy()
        return prepared.reset_index(drop=True)

    train_df = prepare(raw_train, "train", cap_per_skill=5000, require_response=True)
    test_df = prepare(raw_test, "test", cap_per_skill=None, require_response=False)
    counts = train_df.groupby(["skill", "func_id"], as_index=False).size().rename(columns={"size": "n_kept"})
    counts = counts.sort_values(["func_id", "skill"])
    STAGE1_DIR.mkdir(parents=True, exist_ok=True)
    train_df.to_parquet(FILTERED_TRAIN_PARQUET, index=False)
    test_df.to_parquet(FILTERED_TEST_PARQUET, index=False)
    counts.to_csv(STAGE1_STATS_CSV, index=False)
    print("Filtered paper Stage-1 train counts:")
    print(counts.to_string(index=False))
    print(f"Filtered Stage-1 train total: {len(train_df)}")
    print(f"[run] saved {FILTERED_TRAIN_PARQUET}")
    print(f"[run] saved {FILTERED_TEST_PARQUET}")
    print(f"[run] saved {STAGE1_STATS_CSV}")
    return train_df, test_df, counts


def _release_stage1_sft_vllm():
    global _ACTIVE_LLM, _ACTIVE_LLM_KEY
    if _ACTIVE_LLM is not None:
        del _ACTIVE_LLM
        _ACTIVE_LLM = None
        _ACTIVE_LLM_KEY = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def _stage1_sft_training_args(output_dir, learning_rate):
    common = dict(
        output_dir=str(output_dir),
        num_train_epochs=2.0,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=float(learning_rate),
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        save_strategy="epoch",
        report_to=["wandb"],
        run_name="stage1-sft",
    )
    try:
        from trl import SFTConfig
        return SFTConfig(
            **common,
            max_length=int(CFG["max_prompt_len"]) + int(CFG["max_completion_len"]),
            dataset_text_field="text",
            packing=False,
        )
    except Exception:
        from transformers import TrainingArguments
        return TrainingArguments(**common)


def _make_stage1_sft_trainer(model, tokenizer, train_dataset, peft_config, args):
    kwargs = dict(model=model, args=args, train_dataset=train_dataset)
    if peft_config is not None:
        kwargs["peft_config"] = peft_config
    try:
        return SFTTrainer(**kwargs, processing_class=tokenizer)
    except TypeError:
        try:
            return SFTTrainer(
                **kwargs,
                tokenizer=tokenizer,
                dataset_text_field="text",
                max_seq_length=int(CFG["max_prompt_len"]) + int(CFG["max_completion_len"]),
                packing=False,
            )
        except TypeError:
            return SFTTrainer(**kwargs, tokenizer=tokenizer)


def _train_stage1_sft(train_df):
    # Escalation switch if Gate G1 misses: set USE_FULL_FINE_TUNING=True and train all weights
    # at lr=2e-5, matching the paper's full fine-tuning setting instead of LoRA at lr=1e-4.
    USE_FULL_FINE_TUNING = False
    sft_lr = 2e-5 if USE_FULL_FINE_TUNING else 1e-4
    _release_stage1_sft_vllm()
    STAGE1_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    train_dataset = Dataset.from_list([
        {"text": row["stripped_prompt"] + "\n" + row["response"]}
        for row in train_df.to_dict("records")
    ])

    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_name"],
        token=HF_TOKEN,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    model.config.use_cache = False
    peft_config = None if USE_FULL_FINE_TUNING else LoraConfig(
        r=int(CFG["lora_r"]),
        lora_alpha=int(CFG["lora_r"]) * 2,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )

    wandb.init(project="inverse-rl", name="stage1-sft", config={**dict(CFG), "stage1_sft_lr": sft_lr, "stage1_sft_epochs": 2}, reinit=True)
    try:
        args = _stage1_sft_training_args(STAGE1_ADAPTER_DIR, sft_lr)
        trainer = _make_stage1_sft_trainer(model, tokenizer, train_dataset, peft_config, args)
        trainer.train()
        if USE_FULL_FINE_TUNING:
            print(f"[run] saving full fine-tuned Stage-1 model to {STAGE1_DIR}")
            STAGE1_DIR.mkdir(parents=True, exist_ok=True)
            trainer.save_model(str(STAGE1_DIR))
            tokenizer.save_pretrained(STAGE1_DIR)
        else:
            trainer.save_model(str(STAGE1_ADAPTER_DIR))
            print(f"[run] merging Stage-1 SFT adapter and saving merged model to {STAGE1_DIR}")
            merged = trainer.model.merge_and_unload()
            STAGE1_DIR.mkdir(parents=True, exist_ok=True)
            merged.save_pretrained(STAGE1_DIR, safe_serialization=True)
            tokenizer.save_pretrained(STAGE1_DIR)
    finally:
        wandb.finish()
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def _gate_stage1_sft_g1(test_df):
    stage1_model = load_model(STAGE1_DIR)
    prompts = test_df["stripped_prompt"].tolist()
    completions = generate(prompts, n=1, model=stage1_model)
    rows = []
    for row, samples in zip(test_df.to_dict("records"), completions):
        parsed = extract_last_json(samples[0] if samples else "")
        predicted = parsed.get("output") if isinstance(parsed, dict) else None
        reference = _reward_ground_truth(pd.Series(row))
        if reference is not None and not isinstance(reference, str):
            reference = str(reference)
        rows.append({
            "skill": row["skill"],
            "func_id": row["func_id"],
            "correct": float(predicted == reference),
        })
    detail_df = pd.DataFrame(rows)
    skill_df = detail_df.groupby(["skill", "func_id"], as_index=False).agg(accuracy=("correct", "mean"), n=("correct", "size"))
    overall = float(detail_df["correct"].mean()) if len(detail_df) else 0.0
    out_df = pd.concat([
        skill_df.sort_values(["func_id", "skill"]),
        pd.DataFrame([{"skill": "OVERALL", "func_id": "ALL", "accuracy": overall, "n": len(detail_df)}]),
    ], ignore_index=True)
    out_df.to_csv(STAGE1_FORWARD_CSV, index=False)
    print(f"[run] saved {STAGE1_FORWARD_CSV}")
    print(out_df.to_string(index=False))
    bad_skills = skill_df[skill_df["accuracy"] < 0.75].sort_values(["accuracy", "skill"])
    passed = overall >= 0.90 and bad_skills.empty
    print(f"Gate G1: {'PASS' if passed else 'WARN'} — paper test overall accuracy={overall:.3f}")
    if not passed:
        print("Offending skills (accuracy < 0.75):")
        print(bad_skills.to_string(index=False) if len(bad_skills) else "None; overall accuracy missed 0.90")
    state_save("stage1_sft", True)
    state_save("stage1_forward", True)
    return out_df


if STAGE1_DIR.exists():
    print(f"[skip] Cell 4 Stage 1 forward SFT: merged model exists at {STAGE1_DIR}")
    stage1_model = load_model(STAGE1_DIR)
    state_save("stage1_sft", True)
    if STAGE1_FORWARD_CSV.exists():
        state_save("stage1_forward", True)
else:
    print("[run] Cell 4 Stage 1 forward SFT on paper Stage-1 corpus")
    train_df, test_df, stats_df = _load_and_filter_stage1_sft_data()
    _train_stage1_sft(train_df)
    _gate_stage1_sft_g1(test_df)


In [ ]:
# Cell 5 — Stage 1.5 inverse baseline (reversal-curse gate; EXECUTION_GUIDE Step 4)
import json
from pathlib import Path

import pandas as pd

STAGE1_DIR = CKPT_DIR / "stage1_rft"
STAGE1P5_CSV = RESULTS_DIR / "stage1p5_inverse.csv"
STAGE1P5_TIER_CSV = RESULTS_DIR / "stage1p5_inverse_tier_detail.csv"


def _load_stage1_or_fail():
    if not STAGE1_DIR.exists():
        raise FileNotFoundError(f"Stage-1 checkpoint not found: {STAGE1_DIR}; run Cell 4 first")
    return load_model(STAGE1_DIR)


def _level1_inverse_cells():
    return [p for p in _read_jsonl(DATA_DIR / "inv_l1to4_eval.jsonl") if int(p.get("level", 0)) == 1]


def _stage1p5_tier_detail(model, cells, k):
    if STAGE1P5_TIER_CSV.exists():
        print(f"[skip] loading existing Stage-1.5 tier detail from {STAGE1P5_TIER_CSV}")
        return pd.read_csv(STAGE1P5_TIER_CSV)

    print("[run] computing Stage-1.5 per-problem inverse rewards for Gate G2 using hidden-def prompts")
    completions = generate([p["prompt"] for p in cells], n=int(k), model=model)
    rows = []
    for problem, samples in zip(cells, completions):
        rewards = [inverse_reward(sample, problem) for sample in samples]
        skill = problem["chain"][0]
        tier = SKILLS[skill][4]
        rows.append({
            "skill": skill,
            "tier": tier,
            "split": problem.get("eval_split") or ("seen" if problem.get("skills_seen") else "held_out"),
            "level": int(problem.get("level", 0)),
            "pass@1": float(rewards[0] == 1.0) if rewards else 0.0,
            f"pass@{int(k)}": float(any(r == 1.0 for r in rewards)),
        })
    df = pd.DataFrame(rows)
    df.to_csv(STAGE1P5_TIER_CSV, index=False)
    state_save(STAGE1P5_TIER_CSV.stem, True)
    print(f"[run] saved {STAGE1P5_TIER_CSV}")
    return df


def _gate_g2(tier_df):
    hard = tier_df[tier_df["tier"].isin([2, 3, "T2", "T3"])]
    t23_pass1 = float(hard["pass@1"].mean()) if len(hard) else 0.0
    threshold = float(CFG["g2_pass1_threshold"])
    passed = t23_pass1 < threshold
    tier_summary = tier_df.groupby("tier")["pass@1"].mean().sort_index()
    print(f"Gate G2: {'PASS' if passed else 'WARN'} — T2–T3 level-1 inverse pass@1={t23_pass1:.3f} (threshold < {threshold:.3f}); tier means={tier_summary.to_dict()}")
    if not passed:
        print("WARN: reversal-curse contrast is weak; consider restricting the headline analysis to T2–T3.")
    state_save("stage1p5_inverse", True)
    return passed


if STAGE1P5_CSV.exists():
    print(f"[skip] Cell 5 Stage 1.5 inverse baseline: found {STAGE1P5_CSV}")
else:
    print("[run] Cell 5 Stage 1.5 inverse baseline")

stage1_model = _load_stage1_or_fail()
level1_cells = _level1_inverse_cells()
# eval_task uses each problem's canonical hidden-def prompt and writes results/stage1p5_inverse.csv.
stage1p5_df = eval_task(
    model=stage1_model,
    cells=level1_cells,
    task="inverse",
    k=int(CFG["pass_at_k"]),
    out_name="stage1p5_inverse.csv",
)
print(stage1p5_df.to_string(index=False))

tier_detail_df = _stage1p5_tier_detail(stage1_model, level1_cells, int(CFG["pass_at_k"]))
_gate_g2(tier_detail_df)


In [ ]:
# Cell 6 — Stage 2 composition control / shared trainer (stub for EXECUTION_GUIDE Step 5)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 6 stub: Stage 2 composition control / shared trainer (stub for EXECUTION_GUIDE Step 5)")


In [ ]:
# Cell 7 — Stage 2 inversion RL vs RFT (stub for EXECUTION_GUIDE Step 6)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 7 stub: Stage 2 inversion RL vs RFT (stub for EXECUTION_GUIDE Step 6)")


In [ ]:
# Cell 8 — Analysis, plots, and final exports (stub for EXECUTION_GUIDE Step 7)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 8 stub: Analysis, plots, and final exports (stub for EXECUTION_GUIDE Step 7)")
